# 10年定着予測 - AutoGluon（48_ = `47_` から ray を抜いたもの）

## `47_` は失敗した。原因は私が足した `ray`

`47_` では「foldを並列化して時間切れを防ぐ」つもりで
`!pip install "ray>=2.43.0,<2.57.0"` を追加した。これが全モデルを壊した。

```
Warning: Exception caused LightGBMXT_BAG_L1 to fail during training... Skipping this model.
  pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility.
  Expected 112 from C header, got 104 from PyObject

Traceback:
  autogluon/core/models/ensemble/fold_fitting_strategy.py:1264, in _prepare_data
    X = self.ray.put(self.X)
```

ray 2.56.1 の pyarrow が Colab 既存のものと**バイナリ非互換**で、`ray.put()` が必ず落ちる。
ray があると AutoGluon は `ParallelLocalFoldFittingStrategy` を選ぶため、
**k-foldを実際に切る全モデル（CatBoost・LightGBM・XGBoost・NeuralNetTorch）が学習前に全滅**した。

生き残ったのは `use_child_oof=True` で fold戦略を通らない **RandomForest と ExtraTrees だけ**。

| run | `44_` モデル数 | `47_` モデル数 |
|---|---|---|
| lean113_holdout | 149 | **38** |
| lean113_full | 93 | **38** |
| full441_holdout | 63 | **38** |
| full441_full | 41 | **38** |

| | holdout 最良（生存者535名） |
|---|---|
| `44_` lean113 | **0.501305** |
| `47_` lean113 | 0.535162 **（+0.0339 悪化）** |
| `44_` full441 | **0.505552** |
| `47_` full441 | 0.541285 **（+0.0357 悪化）** |

**`47_` の提出ファイル4本は無効。提出しないこと。**

`44_` のログにあった `Will use sequential fold fitting strategy because import of ray failed` は
**警告であって不具合ではなかった**。逐次でも正しく動いていたものを、並列化しようとして壊した。

## `48_` の方針: `47_` から ray だけを抜く

`44_` で見つかった3つの問題のうち、**修正が正しかったのは2つだけ**だった。

| # | 問題 | 判定 | `48_` |
|---|---|---|---|
| 1 | FASTAI全滅（fastai/fastcore不整合） | 時間の浪費のみ | ✅ `excluded_model_types=["FASTAI"]` を維持 |
| 2 | XGBoost 2種が `SettingWithCopyError` | 私のバグ | ✅ `_frame()` の `.copy()` を維持（XGBoostが復活する） |
| 3 | 時間切れ | 一部本物 | ✅ DyStack無効化は維持 / ❌ **ray は撤回** |

`dynamic_stacking=False` の根拠は `44_` 自身のログ ——
`Optimal num_stack_levels=1 (Stacked Overfitting Occurred: False)`。
同じ答えを毎回900秒かけて再発見する必要はない。

## 実行構成

| 特徴量セット | 列数 | 実行 |
|---|---|---|
| `lean113` | 113 | ✅ |
| **`full441`** | **441** | ✅ **本命（`44_` の新最良 0.517685 はこちら）** |

`time_limit = 7200秒/fit`。FASTAI除外とDyStack無効化で浮いた時間を CatBoost 系に回す。
`44_` は441列で **41モデルしか完走していない状態で勝っている**ので、伸びしろがある。

## 実行後にまず確認すること

第D節の `report_failures()` が各runのモデル数とファミリを出力する。

- **CatBoost が0件なら fold戦略が壊れている**（`47_` と同じ症状）。その場合は
  ランタイムを初期化して ray が入っていない状態からやり直す
- XGBoost が載っていれば `SettingWithCopyError` の修正が効いている
- **モデル数が4つのrunで同じ値に揃っていたら異常**。時間切れならrunごとにばらつく

## 判定（事前登録・`44_` と同一）

- 採否は **Public のみ**
- 現最良 `44_ AG_full441_best_single`（**0.517685**）との予測平均絶対差が
  **ノイズ床（`41_` 実測 95%上限 0.02122）** を超えないものは提出しない
- **単体モデルを本命**とする（`13_` では Weighted が単体に0.0012負け、
  `44_` でも提出したのは単体だった）

> ⚠️ **ローカルMacで先行実行しないこと。**
> ⚠️ **`ray` を入れないこと。** 第12節の冒頭でチェックしている。


In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 15.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 23.0 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "48_autogluon_no_ray"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-13 13:38:14] [INFO] === [48_autogluon_no_ray] 実験開始 ===


INFO:48_autogluon_no_ray:=== [48_autogluon_no_ray] 実験開始 ===


[2026-08-13 13:38:14] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260813


INFO:48_autogluon_no_ray:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260813


[2026-08-13 13:38:14] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/48_autogluon_no_ray_checkpoint.csv


INFO:48_autogluon_no_ray:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/48_autogluon_no_ray_checkpoint.csv


[2026-08-13 13:38:14] [INFO] チェックポイントは未作成（新規実行）


INFO:48_autogluon_no_ray:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-13 13:38:19] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:48_autogluon_no_ray:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-13 13:38:19] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:48_autogluon_no_ray:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-13 13:38:19] [INFO] 定着率: 0.5647


INFO:48_autogluon_no_ray:定着率: 0.5647


[2026-08-13 13:38:19] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:48_autogluon_no_ray:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-13 13:38:19] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:48_autogluon_no_ray:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-13 13:38:19] [INFO] Test  早期退職者: 0名 / 2502名


INFO:48_autogluon_no_ray:Test  早期退職者: 0名 / 2502名


[2026-08-13 13:38:19] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:48_autogluon_no_ray:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-13 13:38:19] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:48_autogluon_no_ray:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-13 13:38:20] [INFO] ------------------------------------------------------------


INFO:48_autogluon_no_ray:------------------------------------------------------------


[2026-08-13 13:38:20] [INFO] split非依存の基本特徴量を生成中...


INFO:48_autogluon_no_ray:split非依存の基本特徴量を生成中...


[2026-08-13 13:38:20] [INFO] ------------------------------------------------------------


INFO:48_autogluon_no_ray:------------------------------------------------------------


[2026-08-13 13:45:51] [INFO] split非依存の基本特徴量生成完了


INFO:48_autogluon_no_ray:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-13 13:45:52] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:48_autogluon_no_ray:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-13 13:45:53] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:48_autogluon_no_ray:入社時メモ: SVD累積寄与率=0.760


[2026-08-13 13:45:58] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:48_autogluon_no_ray:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-13 13:46:01] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:48_autogluon_no_ray:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-13 13:46:01] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:48_autogluon_no_ray:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-13 13:46:01] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:48_autogluon_no_ray:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-13 13:48:51] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:48_autogluon_no_ray:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-13 13:48:52] [INFO] Persona単位の基本特徴量を生成中...


INFO:48_autogluon_no_ray:Persona単位の基本特徴量を生成中...


[2026-08-13 13:48:52] [INFO] Persona単位の基本特徴量処理完了


INFO:48_autogluon_no_ray:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-13 13:48:52] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:48_autogluon_no_ray:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-13 13:48:52] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:48_autogluon_no_ray:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-13 13:48:52] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:48_autogluon_no_ray:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [15]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [16]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [17]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-13 13:48:53] [INFO] ============================================================


INFO:48_autogluon_no_ray:============================================================


[2026-08-13 13:48:53] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:48_autogluon_no_ray:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-13 13:48:53] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:48_autogluon_no_ray:B/C: train=2208, val=535（生存者のみ）


[2026-08-13 13:48:54] [INFO] D: train=2761（全件）, val=0（空）


INFO:48_autogluon_no_ray:D: train=2761（全件）, val=0（空）


[2026-08-13 13:48:54] [INFO] 特徴量数: 441


INFO:48_autogluon_no_ray:特徴量数: 441


## 10. 特徴量グループの棚卸し（`40_` から移植）

In [18]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


In [19]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## 11. 特徴量セットと AutoGluon の設定

In [20]:
!pip install -q autogluon.tabular
# ⚠️ ray は入れない。47_ では ray 2.56.1 の pyarrow が Colab のものと
#    バイナリ非互換になり、ParallelLocalFoldFittingStrategy 経由で fold を切る
#    全モデル（CatBoost/LightGBM/XGBoost/NN）が学習前に落ちた。
#    ray が無ければ SequentialLocalFoldFittingStrategy に戻り、44_ と同じ挙動になる。

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 127.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.5 MB/s eta 0:00:00


In [21]:
# ============================================================
# 設定
# ============================================================
import pandas as pd
# 問題2の対策その2: AutoGluon内部の代入で SettingWithCopyError を出さない
pd.set_option("mode.chained_assignment", None)

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}
FEATURE_SETS = {
    "lean113": {"groups": CORE_GROUPS, "agg_stats": AGG_KEEP_STATS},   # 40_ R6_lean = 現最良
    "full441": {"groups": ALL_GROUPS,  "agg_stats": None},             # 37_ D3
}
RUN_FULL441 = True   # 44_で時間切れ、かつ現最良ではないので既定ではスキップ
if not RUN_FULL441:
    del FEATURE_SETS["full441"]


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


_EXPECT = {"lean113": 113, "full441": 441}
for _n, _s in FEATURE_SETS.items():
    _c = cols_for(_s, ag_full)
    assert _c == cols_for(_s, ag_train_80b), f"{_n}: 80%学習と全件で列が食い違う"
    assert len(_c) == _EXPECT[_n], f"{_n}: {len(_c)}列（{_EXPECT[_n]}列のはず）"
    print(f"  {_n}: {len(_c)} 列 ✅")

# --- AutoGluon の設定（44_からの修正点） ---
PRESETS = "best_quality"
TIME_LIMIT = 7200            # 秒/fit。FASTAI除外とDyStack無効化のぶんを学習に回す
AG_METRIC = "log_loss"

# 問題1の対策: FASTAI系は環境の不具合で1つ残らず失敗するので最初から除外
EXCLUDED_MODELS = ["FASTAI"]

# 問題3の対策: DyStackを切る。44_自身が「Optimal num_stack_levels=1」と判定済み
DYNAMIC_STACKING = False
NUM_STACK_LEVELS = 1
NUM_BAG_FOLDS = 8

NOISE_FLOOR_P95 = 0.02122    # 41_ 実測のノイズ床

print(f"\npresets={PRESETS} / time_limit={TIME_LIMIT}秒/fit / eval_metric={AG_METRIC}")
print(f"除外モデル: {EXCLUDED_MODELS}（環境の不具合で全滅するため）")
print(f"dynamic_stacking={DYNAMIC_STACKING}, num_stack_levels={NUM_STACK_LEVELS}, "
      f"num_bag_folds={NUM_BAG_FOLDS}")
print(f"提出ゲート: 現最良との予測平均絶対差 > {NOISE_FLOOR_P95}")


  lean113: 113 列 ✅
  full441: 441 列 ✅

presets=best_quality / time_limit=7200秒/fit / eval_metric=log_loss
除外モデル: ['FASTAI']（環境の不具合で全滅するため）
dynamic_stacking=False, num_stack_levels=1, num_bag_folds=8
提出ゲート: 現最良との予測平均絶対差 > 0.02122


## 12. AutoGluon の学習関数（`44_` からの修正込み）

In [22]:
# ============================================================
# AutoGluon の学習関数（44_からの修正込み）
# ============================================================
from autogluon.tabular import TabularPredictor

# 47_ の教訓: ray があると ParallelLocalFoldFittingStrategy が選ばれ、
# pyarrow のバイナリ非互換で fold を切る全モデルが落ちる。無いのが正しい状態。
try:
    import ray  # noqa: F401
    print("⚠️ ray が入っている。47_ ではこれが原因で CatBoost/LightGBM/XGBoost/NN が全滅した。")
    print("   ランタイムを初期化し、ray を入れずにやり直すこと。")
except ImportError:
    print("✅ ray は入っていない（正常）。foldは逐次学習される。")

AG_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
AG_DIR.mkdir(parents=True, exist_ok=True)


def _frame(df, feats, with_label=True):
    """AutoGluon に渡すフレーム。

    44_からの修正: 明示的に .copy() する。
    reset_index だけだと元フレームのスライス由来という _is_copy の印が残ることがあり、
    AutoGluon の XGBoost 前処理が列を代入したときに SettingWithCopyError になっていた。
    -999 埋めをしない点は 44_ と同じ（AutoGluonが自前で前処理する）。
    """
    cols = list(feats) + ([TARGET_COL] if with_label and TARGET_COL in df.columns else [])
    out = df[cols].copy()
    out.reset_index(drop=True, inplace=True)
    out._is_copy = None          # 念のため印を明示的に消す
    return out


def fit_autogluon(tag, train_df, feats, tuning_df=None, time_limit=TIME_LIMIT):
    """AutoGluon を1回 fit する。保存済みなら読み込むだけ（中断に備える）"""
    path = AG_DIR / tag
    if (path / "predictor.pkl").exists():
        logger.info(f"[{tag}] 保存済みpredictorを読み込む（再学習しない）")
        return TabularPredictor.load(str(path))

    logger.info("=" * 60)
    logger.info(f"[{tag}] AutoGluon fit: n={len(train_df)}, 特徴量={len(feats)}, "
                f"time_limit={time_limit}, excluded={EXCLUDED_MODELS}")
    kw = dict(presets=PRESETS, time_limit=time_limit,
              excluded_model_types=EXCLUDED_MODELS,
              num_bag_folds=NUM_BAG_FOLDS, num_stack_levels=NUM_STACK_LEVELS)
    if tuning_df is not None:
        kw["tuning_data"] = _frame(tuning_df, feats)
        kw["use_bag_holdout"] = True
        # use_bag_holdout=True のときAutoGluonはDyStackを自動で切る（44_ログで確認済み）
    else:
        kw["dynamic_stacking"] = DYNAMIC_STACKING
    p = TabularPredictor(label=TARGET_COL, eval_metric=AG_METRIC, path=str(path),
                         problem_type="binary")
    p.fit(_frame(train_df, feats), **kw)
    return p


def leaderboard(predictor, data=None):
    """AutoGluonのバージョン差（silent引数の有無）を吸収する"""
    try:
        return predictor.leaderboard(data, silent=True) if data is not None \
            else predictor.leaderboard(silent=True)
    except TypeError:
        return predictor.leaderboard(data) if data is not None else predictor.leaderboard()


def positive_proba_model(predictor, X, model=None):
    pp = predictor.predict_proba(X, model=model)
    pos = predictor.positive_class if hasattr(predictor, "positive_class") else None
    if pos is None or pos not in pp.columns:
        pos = 1 if 1 in pp.columns else pp.columns[-1]
    return pp[pos].values


def report_failures(predictor, tag):
    """学習に失敗したモデルが残っていないかを確認する。

    44_ では FASTAI が全滅・XGBoost の一部が例外で落ちていた。
    47_ の修正が効いていれば、XGBoost系がleaderboardに載るはず。
    """
    lb = leaderboard(predictor)
    names = list(lb["model"])
    n_xgb = sum(1 for m in names if m.startswith("XGBoost"))
    n_fastai = sum(1 for m in names if "FastAI" in m)
    print(f"  [{tag}] 学習できたモデル {len(names)}件 / うち XGBoost {n_xgb}件")
    if n_fastai:
        print(f"  ⚠️ FASTAIが{n_fastai}件混ざっている（除外できていない）")
    if n_xgb == 0:
        print("  ⚠️ XGBoostが1件も無い。SettingWithCopyErrorの修正が効いていない可能性")
    else:
        print("  ✅ XGBoostが学習できている（44_の SettingWithCopyError は解消）")
    return lb


print("✅ AutoGluon 関数定義完了（copy修正 / FASTAI除外 / DyStack無効）")


✅ ray は入っていない（正常）。foldは逐次学習される。
✅ AutoGluon 関数定義完了（copy修正 / FASTAI除外 / DyStack無効）


## D. 実行

In [23]:
# ============================================================
# 実行
# ============================================================

RESULT_SCHEMA = ["config", "feature_set", "n_features", "run", "model",
                 "val_logloss", "n_train", "pred_mean", "mad_vs_best", "corr_vs_best",
                 "submission_path"]

# 現最良は 44_ の AutoGluon 441列（Public 0.517685）に更新された
_best_files = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_44_autogluon_on_40_pipeline_AG_full441_best_single.csv"))
assert _best_files, "現最良（44_ AG_full441_best_single）の提出ファイルが見つからない"
BEST_PRED = pd.read_csv(_best_files[-1], header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]
BEST_VAL = 0.505552   # 44_ full441 holdout の最良（生存者535名）
print(f"比較基準: {_best_files[-1].name}（予測平均 {BEST_PRED.mean():.4f}, Public 0.517685, val {BEST_VAL}）")

results, leaderboards = {}, {}

for set_name, spec in FEATURE_SETS.items():
    feats = cols_for(spec, ag_full)

    # --- holdout run: 既存構成と同じ検証セットで比較する ---
    p_hold = fit_autogluon(f"{set_name}_holdout", ag_train_80b, feats, tuning_df=ag_val_surv)
    lb = leaderboard(p_hold, _frame(ag_val_surv, feats))
    leaderboards[f"{set_name}_holdout"] = lb
    print(f"\n===== [{set_name}] holdout leaderboard（検証=生存者{len(ag_val_surv)}名）=====")
    print(lb[["model", "score_test", "score_val", "fit_time"]].head(12).to_string(index=False))
    report_failures(p_hold, f"{set_name}_holdout")

    lb = lb.copy()
    lb["val_logloss"] = -lb["score_test"]
    _best_ag = lb["val_logloss"].min()
    print(f"\n  AutoGluonの最良: {_best_ag:.6f} / 44_ full441 holdout 最良: {BEST_VAL:.6f}"
          f"  差 {_best_ag - BEST_VAL:+.6f}")
    print(f"  → {'分解能±0.011を超える差' if abs(_best_ag - BEST_VAL) > 0.011 else '分解能以下（判定不能）'}")
    print("  ※ [validation-asymmetry] 検証が『改善』と言った5件のうちPublicで的中は1件のみ。"
          "Publicで確かめるまで信用しない。")

    # --- full run: 提出用（Train全件） ---
    p_full = fit_autogluon(f"{set_name}_full", ag_full, feats, tuning_df=None)
    lb_full = leaderboard(p_full)
    leaderboards[f"{set_name}_full"] = lb_full
    print(f"\n===== [{set_name}] full leaderboard（AutoGluon内部検証）=====")
    print(lb_full[["model", "score_val", "fit_time"]].head(12).to_string(index=False))
    report_failures(p_full, f"{set_name}_full")

    Xte = _frame(test_features_full, feats, with_label=False)
    cand = {"weighted": lb_full[lb_full["model"].str.startswith("WeightedEnsemble")]["model"].tolist(),
            "best_single": lb_full[~lb_full["model"].str.startswith("WeightedEnsemble")]["model"].tolist()}
    for kind, names in cand.items():
        if not names:
            print(f"  [{set_name}] {kind}: 該当モデルなし。スキップ")
            continue
        mdl = names[0]
        preds = positive_proba_model(p_full, Xte, mdl)
        label = f"AG48_{set_name}_{kind}"
        path = save_submission(test_features_full.index, preds, label)
        p_al = pd.Series(preds, index=test_features_full.index).loc[BEST_PRED.index]
        results[label] = make_row(
            config=label, feature_set=set_name, n_features=len(feats), run="full", model=mdl,
            val_logloss=float(-lb_full.set_index("model").loc[mdl, "score_val"]),
            n_train=len(ag_full), pred_mean=float(preds.mean()),
            mad_vs_best=float(np.abs(p_al.values - BEST_PRED.values).mean()),
            corr_vs_best=float(np.corrcoef(p_al.values, BEST_PRED.values)[0, 1]),
            submission_path=path)
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label}_testpreds.npy", preds)

pd.DataFrame(list(results.values())).to_csv(CHECKPOINT_PATH, index=False)
for k, lb_ in leaderboards.items():
    lb_.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_leaderboard_{k}.csv", index=False)
logger.info("結果とleaderboardを保存")


比較基準: 20260812_44_autogluon_on_40_pipeline_AG_full441_best_single.csv（予測平均 0.5823, Public 0.517685, val 0.505552）
[2026-08-13 13:49:13] [INFO] ============================================================


INFO:48_autogluon_no_ray:============================================================


[2026-08-13 13:49:13] [INFO] [lean113_holdout] AutoGluon fit: n=2208, 特徴量=113, time_limit=7200, excluded=['FASTAI']


INFO:48_autogluon_no_ray:[lean113_holdout] AutoGluon fit: n=2208, 特徴量=113, time_limit=7200, excluded=['FASTAI']
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          8
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       49.15 GB / 50.99 GB (96.4%)
Disk Space Avail:   194.70 GB / 225.83 GB (86.2%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "/content/drive/MyDrive/jaggle_2026/saved_models/

[1000]	valid_set's binary_logloss: 0.535486
[1000]	valid_set's binary_logloss: 0.494895
[1000]	valid_set's binary_logloss: 0.55365


	-0.5345	 = Validation score   (-log_loss)
	52.2s	 = Training   runtime
	0.78s	 = Validation runtime
Fitting model: RandomForest_r39_BAG_L1 ... Training model for up to 3611.23s of the 6012.34s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=8, gpus=0, mem=0.0/48.7 GB
	-0.5746	 = Validation score   (-log_loss)
	7.17s	 = Training   runtime
	0.12s	 = Validation runtime
Fitting model: CatBoost_r167_BAG_L1 ... Training model for up to 3603.61s of the 6004.73s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.5132	 = Validation score   (-log_loss)
	50.8s	 = Training   runtime
	0.15s	 = Validation runtime
Fitting model: NeuralNetTorch_r41_BAG_L1 ... Training model for up to 3552.31s of the 5953.42s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.5646	 = Validation score   

KeyboardInterrupt: 

## E. 結果まとめ

In [ ]:
# ============================================================
# 結果まとめと提出判定
# ============================================================

summary = pd.DataFrame(list(results.values()))
summary["提出"] = np.where(summary["mad_vs_best"] > NOISE_FLOOR_P95, "提出する",
                            "見送り（現最良との差がノイズ床以下）")
pd.set_option("display.width", 240)
print(summary[["config", "feature_set", "n_features", "model", "val_logloss",
               "pred_mean", "mad_vs_best", "corr_vs_best", "提出"]].round(6).to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)

print()
print("=" * 74)
print("提出候補（単体モデルを優先。13_ では単体が Weighted に0.0012勝っている）")
print("=" * 74)
_order = [c for c in summary["config"] if c.endswith("best_single")] + \
         [c for c in summary["config"] if c.endswith("weighted")]
_i = 0
for c in _order:
    r = summary[(summary["config"] == c) & (summary["提出"] == "提出する")]
    if len(r) == 0:
        continue
    _i += 1
    r = r.iloc[0]
    print(f"{_i}. {c:<28s} {Path(r['submission_path']).name}")
    print(f"     {r['feature_set']} {int(r['n_features'])}列 / model={r['model']}")
    print(f"     現最良との相関 {r['corr_vs_best']:.5f} / 平均絶対差 {r['mad_vs_best']:.5f} "
          f"/ 予測平均 {r['pred_mean']:.4f}")
if _i == 0:
    print("  なし。AutoGluonの予測が現最良からノイズ床以上に離れなかった。")

print()
print("=" * 74)
print("44_ が出力した提出ファイルとの比較（同じ設定で環境不具合だけが違う）")
print("=" * 74)
_old = sorted((PROJECT_ROOT / "data" / "output").glob("*/*_44_autogluon_on_40_pipeline_AG_lean113_*.csv"))
_old = [f for f in _old if "summary" not in f.name]
if _old:
    for f in _old:
        s = pd.read_csv(f, header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"].loc[BEST_PRED.index]
        line = f"  {f.name}: 予測平均 {s.mean():.4f}"
        for c in summary["config"]:
            kind = "weighted" if c.endswith("weighted") else "best_single"
            if kind in f.name:
                n = pd.read_csv(summary.set_index("config").loc[c, "submission_path"],
                                header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"].loc[BEST_PRED.index]
                line += f" / 47_{kind}との平均絶対差 {np.abs(s.values - n.values).mean():.5f}"
        print(line)
    print("  → 差が小さければ、FASTAI/XGBoostの失敗はスコアに影響していなかったことになる。")
else:
    print("  44_ の提出ファイルが見つからなかった")


## 提出方針

第E節で「提出する」となったものを、**単体モデル → WeightedEnsemble の順**に提出する。

### `44_` の出力をどう扱うか

`44_` の `lean113` 側は**完走しており、2ファイルとも提出可能**である
（現最良との平均絶対差 0.047、ノイズ床0.021を大きく超える）。

- **`44_` の結果をそのまま提出してよい。** FASTAIとXGBoost一部の失敗は
  「使えるモデルが少し減った」だけで、出力が壊れているわけではない
- `47_` はその上で「環境不具合を除いたらどうなるか」を測るもの。
  両者の予測差が小さければ、失敗したモデルはもともと選ばれていなかったと分かる

### 結果の解釈ルール（事前登録）

- **Public < 0.521729** → AutoGluonのモデル空間に価値がある。中身を見て次を決める
- **Public ≒ 0.5217 ± 0.001** → 差なし。CatBoost単体を基準構成のまま維持
- **Public > 0.5217** → `13_` と同じ結論。この線を打ち切る
- **WeightedEnsemble が単体に負けたら** → `12_`・`13_` に続き3度目。
  「OOFから重みを学習すると過学習する」を確定させる

### 検証スコアの扱い

`44_` の holdout で AutoGluon の `WeightedEnsemble_L3` が **0.501305**、
`40_` R6_lean（CatBoost 8シード平均）が **0.514642** で、差 **−0.0133** は分解能±0.011を超えた。
本プロジェクトで検証が現行構成を明確に上回った初の事例である。

**それでも提出の根拠にはしない。** [validation-asymmetry] のとおり、
検証が「改善」と言った5件のうち Public で的中したのは1件だけである。
根拠に使うのはラベルを見ない指標（現最良との予測差がノイズ床0.02122を超えるか）に限る。

### 関連

- 現最良: `data/output/20260811/20260811_40_feature_reduction_R6_lean.csv`（Public 0.521729）
- `44_` の実行ログと問題: 本ノートブック冒頭
- 前回の AutoGluon: `submit_result_report.md` 第9〜10節（`13_`）
